# Population and income features

Add population (WorldPop) and income-proxy layers (Meta's Relative Wealth Index and VIIRS
nighttime lights) at both the station locations and the analysis grid cells from notebook 04
(Section 5.4 / 5.5, Phase F). These are exposure and equity context, not causes of pollution,
so they are kept as separate columns that can be tested for predictive value later.

In [8]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Imports

In [9]:
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import rioxarray
from rasterio.enums import Resampling
import xarray as xr
from scipy.spatial import cKDTree
import ee
import dotenv
import yaml

import monitoring_deserts.data.fetch_population as fetch_pop

## Configuration

In [10]:
# Generic settings
with open("../configs/project_config.yaml") as f:
    gen_config = yaml.safe_load(f)

RAW_DATA_DIR = Path(gen_config["data_dir"]["raw"]) / "population"
CLEAN_DATA_DIR = Path(gen_config["data_dir"]["clean"]) / "population"
PROCESSED_DATA_DIR = Path(gen_config["data_dir"]["processed"]) # The data/processed/ should have no subdirs
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
CLEAN_DATA_DIR.mkdir(parents=True, exist_ok=True)

EQUAL_AREA_CRS = gen_config["crs"]["equal_area"]

# Spacetime settings
with open("../configs/census_scope.yaml") as f:
    census_config = yaml.safe_load(f)

BOUNDING_BOX = tuple(census_config["bounding_box"].values())
DAY_START, DAY_END = census_config["period"]["start"], census_config["period"]["end"]

# Countries to download WorldPop / RWI for (ISO 3166-1 alpha-3). Note that BOUNDING_BOX
# also touches countries outside this list (e.g. Afghanistan, Bhutan), so grid cells
# there will have no population/RWI values unless they are added.
COUNTRIES = ["IND", "PAK", "BGD", "NPL", "LKA"]

# Open the analysis df (stations x months, with satellite and meteorology features) from
# notebook 5, and the analysis grid from notebook 4
analysis_df = pd.read_parquet(CLEAN_DATA_DIR.parent / "meteorology" / "analysis_df.parquet")
analysis_grid = gpd.read_parquet(PROCESSED_DATA_DIR / "grid.parquet")

# Initialize Earth Engine client (for VIIRS nighttime lights)
dotenv.load_dotenv()
ee.Initialize(project="monitoring-deserts")

## Step 1 - Download population data

In [4]:
# Download the population data
paths = fetch_pop.pull_worldpop_countries(COUNTRIES, RAW_DATA_DIR, year=2020, resolution="1km")

[1/5] IND: ind_ppp_2020_1km_Aggregated_UNadj.tif already downloaded, skipping


[2/5] PAK:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

[3/5] BGD:   0%|          | 0.00/824k [00:00<?, ?B/s]

[4/5] NPL:   0%|          | 0.00/867k [00:00<?, ?B/s]

[5/5] LKA:   0%|          | 0.00/364k [00:00<?, ?B/s]

## Step 2 - Mosaic the country rasters

In [11]:
# Merge the per-country rasters into one region-wide raster, clipped to the study bounding box
pop_raster = fetch_pop.mosaic_rasters(paths, bbox=BOUNDING_BOX)

# Sanity check: shape, CRS, cell size, and total population inside the bounding box
print(pop_raster.shape, pop_raster.rio.crs, pop_raster.rio.resolution())
print(f"Total population in mosaic: {float(pop_raster.sum()):,.0f}")

(3742, 3898) EPSG:4326 (0.008333333299999999, -0.0083333333)
Total population in mosaic: 1,800,289,280


## Step 3 - Population onto the analysis grid

Population is a count, so source pixels are **summed** into each grid cell (Appendix C.7), not averaged.

In [12]:
# Build an empty raster whose cells are exactly the analysis-grid cells. The grid is regular,
# so its unique centroid longitudes/latitudes give the raster's x/y axes. The centroids are
# rounded because shapely computes them with floating-point noise (~1e-14), which would
# otherwise split each real longitude/latitude into several "unique" values.
grid_lons = np.sort(analysis_grid["centroid_lon"].round(9).unique())
grid_lats = np.sort(analysis_grid["centroid_lat"].round(9).unique())
assert len(grid_lons) * len(grid_lats) == len(analysis_grid), "analysis_grid is not a full regular grid"

template = xr.DataArray(
    np.zeros((len(grid_lats), len(grid_lons)), dtype="float32"),
    coords={"y": grid_lats[::-1], "x": grid_lons},  # rasters are north-up, so y descends
    dims=("y", "x"),
).rio.write_crs(gen_config["crs"]["storage"])

In [13]:
# Resample onto the grid, summing the source pixels that fall in each cell
pop_gridded = pop_raster.rio.reproject_match(template, resampling=Resampling.sum)

# Conservation check: the gridded total should be close to the mosaic's total
src_total = float(pop_raster.sum())
grid_total = float(pop_gridded.sum())
print(f"Mosaic total:  {src_total:,.0f}")
print(f"Gridded total: {grid_total:,.0f}")
print(f"Ratio: {grid_total / src_total:.4f}")

Mosaic total:  1,800,289,280
Gridded total: 1,800,289,152
Ratio: 1.0000


In [14]:
# Read each cell's value back by (row, col), so it is keyed on cell_id rather than row order
col = np.searchsorted(grid_lons, analysis_grid["centroid_lon"].round(9).values)
row = len(grid_lats) - 1 - np.searchsorted(grid_lats, analysis_grid["centroid_lat"].round(9).values)  # y descends

grid_population = pd.DataFrame({
    "cell_id": analysis_grid["cell_id"].values,
    "population": pop_gridded.values[row, col],
})

print(f"{grid_population['population'].isna().sum():,} of {len(grid_population):,} cells have no population data")
grid_population["population"].describe()

279,850 of 434,940 cells have no population data


count    1.550900e+05
mean     1.160803e+04
std      3.293594e+04
min      0.000000e+00
25%      1.565865e+03
50%      5.472563e+03
75%      1.253669e+04
max      2.500372e+06
Name: population, dtype: float64